# OECD FDI Income: Bayesian Re-Estimation and Feature Creation

This notebook recalculates the key modeling outcome with a Bayesian-style empirical updating logic.

The goal is practical rather than purely theoretical:

- build posterior candidate values for the transformed outcome
- compare which candidate source is more probable
- create new variables from the most probable posterior values
- use those new variables in the final core model


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display, Markdown


C:\Users\Yusuf\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent

data_path = project_root / 'data' / 'processed' / 'OECD_FDI_Except_Pointless.csv'
df = pd.read_csv(data_path, sep=';', encoding='latin1', engine='python', on_bad_lines='skip')
df['OBS_VALUE_NUM'] = pd.to_numeric(
    df['OBS_VALUE'].astype(str).str.replace('.', '', regex=False),
    errors='coerce'
)
df['obs_missing_flag'] = df['OBS_VALUE_NUM'].isna().astype(int)
df['obs_value_imputed'] = df['OBS_VALUE_NUM'].fillna(df['OBS_VALUE_NUM'].median())
df['obs_value_signed_log'] = np.sign(df['obs_value_imputed']) * np.log1p(np.abs(df['obs_value_imputed']))

q1 = df['obs_value_signed_log'].quantile(0.25)
q3 = df['obs_value_signed_log'].quantile(0.75)
iqr = q3 - q1
df['obs_outlier_signed_log_flag'] = ((df['obs_value_signed_log'] < q1 - 1.5 * iqr) | (df['obs_value_signed_log'] > q3 + 1.5 * iqr)).astype(int)
df[['obs_value_signed_log', 'obs_missing_flag', 'obs_outlier_signed_log_flag']].head()


,obs_value_signed_log,obs_missing_flag,obs_outlier_signed_log_flag
0,18.327907,1,0
1,30.596330,0,0
2,30.627683,0,0
3,32.209225,0,0
4,18.327907,1,0


## Empirical Bayes Candidate Values

Posterior means are built for several grouping structures. Each candidate is a shrinkage estimate between the group mean and the global prior mean.


In [3]:
global_mean = df['obs_value_signed_log'].mean()
global_var = df['obs_value_signed_log'].var(ddof=1)

def empirical_bayes_mean(data, group_col, value_col='obs_value_signed_log'):
    grouped = data.groupby(group_col)[value_col]
    stats_df = grouped.agg(['mean', 'count', 'var']).reset_index()
    between_var = max(stats_df['mean'].var(ddof=1), 1e-6)
    within_var = max(stats_df['var'].fillna(global_var).mean(), 1e-6)
    stats_df['shrinkage_weight'] = (stats_df['count'] * between_var) / (stats_df['count'] * between_var + within_var)
    stats_df['posterior_mean'] = stats_df['shrinkage_weight'] * stats_df['mean'] + (1 - stats_df['shrinkage_weight']) * global_mean
    return stats_df[[group_col, 'posterior_mean', 'count', 'shrinkage_weight']]


ref_area_post = empirical_bayes_mean(df, 'REF_AREA').rename(columns={'posterior_mean': 'bayes_ref_area'})
activity_post = empirical_bayes_mean(df, 'ACTIVITY').rename(columns={'posterior_mean': 'bayes_activity'})
type_entity_post = empirical_bayes_mean(df, 'TYPE_ENTITY').rename(columns={'posterior_mean': 'bayes_type_entity'})

df = df.merge(ref_area_post[['REF_AREA', 'bayes_ref_area']], on='REF_AREA', how='left')
df = df.merge(activity_post[['ACTIVITY', 'bayes_activity']], on='ACTIVITY', how='left')
df = df.merge(type_entity_post[['TYPE_ENTITY', 'bayes_type_entity']], on='TYPE_ENTITY', how='left')
df['bayes_global'] = global_mean

df[['bayes_ref_area', 'bayes_activity', 'bayes_type_entity', 'bayes_global']].head()


,bayes_ref_area,bayes_activity,bayes_type_entity,bayes_global
0,15.391915,14.979001,6.802485,15.692267
1,19.090753,16.401306,20.102650,15.692267
2,19.090753,13.956537,20.102650,15.692267
3,15.524392,15.163081,20.171667,15.692267
4,19.090753,16.401306,20.171667,15.692267


## Posterior Probability Comparison and MAP Choice

For each observation, the notebook compares candidate posterior values and keeps the most probable one under a Gaussian approximation on the transformed scale.


In [4]:
sigma2 = max(df['obs_value_signed_log'].var(ddof=1), 1e-6)
candidate_cols = ['bayes_ref_area', 'bayes_activity', 'bayes_type_entity', 'bayes_global']

for col in candidate_cols:
    df[f'loglik_{col}'] = -0.5 * ((df['obs_value_signed_log'] - df[col]) ** 2) / sigma2

loglik_cols = [f'loglik_{col}' for col in candidate_cols]
max_loglik = df[loglik_cols].max(axis=1)
exp_shifted = np.exp(df[loglik_cols].sub(max_loglik, axis=0))
posterior_probs = exp_shifted.div(exp_shifted.sum(axis=1), axis=0)
posterior_probs.columns = [col.replace('loglik_', 'posterior_prob_') for col in posterior_probs.columns]
df = pd.concat([df, posterior_probs], axis=1)

posterior_prob_cols = posterior_probs.columns.tolist()
df['bayes_best_source'] = df[posterior_prob_cols].idxmax(axis=1).str.replace('posterior_prob_', '', regex=False)
df['bayes_best_probability'] = df[posterior_prob_cols].max(axis=1)
df['obs_value_bayes_selected_signed_log'] = np.select(
    [
        df['bayes_best_source'] == 'bayes_ref_area',
        df['bayes_best_source'] == 'bayes_activity',
        df['bayes_best_source'] == 'bayes_type_entity',
    ],
    [df['bayes_ref_area'], df['bayes_activity'], df['bayes_type_entity']],
    default=df['bayes_global']
)
df['obs_value_bayes_selected_rawscale'] = np.sign(df['obs_value_bayes_selected_signed_log']) * np.expm1(np.abs(df['obs_value_bayes_selected_signed_log']))
df[['bayes_best_source', 'bayes_best_probability', 'obs_value_bayes_selected_signed_log', 'obs_value_bayes_selected_rawscale']].head()


,bayes_best_source,bayes_best_probability,obs_value_bayes_selected_signed_log,obs_value_bayes_selected_rawscale
0,bayes_global,0.266339,15.692267,6.532283e+06
1,bayes_type_entity,0.280195,20.102650,5.376133e+08
2,bayes_type_entity,0.290134,20.102650,5.376133e+08
3,bayes_type_entity,0.307263,20.171667,5.760280e+08
4,bayes_ref_area,0.251623,19.090753,1.954378e+08


## Use Bayesian Variables to Define the Final Core Model

The Bayesian-selected transformed outcome is used as the main response variable for the final model-building step.


In [5]:
bayes_model_df = df[[
    'obs_value_bayes_selected_signed_log', 'TYPE_ENTITY', 'MEASURE_PRINCIPLE', 'TIME_PERIOD',
    'obs_missing_flag', 'obs_outlier_signed_log_flag', 'bayes_best_probability'
]].dropna().copy()

bayes_formula = (
    'obs_value_bayes_selected_signed_log ~ C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD '
    '+ obs_missing_flag + obs_outlier_signed_log_flag + bayes_best_probability'
)
bayes_model = smf.ols(bayes_formula, data=bayes_model_df).fit(cov_type='HC3')
print(bayes_model.summary())


                                     OLS Regression Results                                    
Dep. Variable:     obs_value_bayes_selected_signed_log   R-squared:                       0.563
Model:                                             OLS   Adj. R-squared:                  0.563
Method:                                  Least Squares   F-statistic:                     1988.
Date:                                 Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                                         19:06:48   Log-Likelihood:                -28608.
No. Observations:                                10344   AIC:                         5.723e+04
Df Residuals:                                    10337   BIC:                         5.728e+04
Df Model:                                            6                                         
Covariance Type:                                   HC3                                         
                                  coef  

C:\Users\Yusuf\AppData\Roaming\Python\Python313\site-packages\statsmodels\regression\linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])
C:\Users\Yusuf\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 7, but rank is 6
  warnings.warn('covariance of constraints does not have full '


In [6]:
df['bayes_model_fitted_signed_log'] = bayes_model.predict(df)
df['bayes_model_residual_signed_log'] = df['obs_value_bayes_selected_signed_log'] - df['bayes_model_fitted_signed_log']
df['bayes_model_fitted_rawscale'] = np.sign(df['bayes_model_fitted_signed_log']) * np.expm1(np.abs(df['bayes_model_fitted_signed_log']))
df[['bayes_model_fitted_signed_log', 'bayes_model_residual_signed_log', 'bayes_model_fitted_rawscale']].head()


,bayes_model_fitted_signed_log,bayes_model_residual_signed_log,bayes_model_fitted_rawscale
0,13.988539,1.703728,1.188899e+06
1,18.545158,1.557492,1.132557e+08
2,18.197401,1.905249,7.998908e+07
3,16.989282,3.182384,2.389745e+07
4,21.482012,-2.391259,2.135596e+09


### Comment

The final modeling variable is `obs_value_bayes_selected_signed_log`. It represents the most probable posterior candidate among the country-based, activity-based, entity-based, and global shrinkage estimates. This turns the Bayesian comparison into a directly usable modeling variable rather than leaving it as a purely descriptive posterior exercise.
